### RAG Pipeline - Data Ingestion to Vector DB

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/Users/aaronchen/Documents/Code/learning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf documents in the directory
def process_all_pdfs(pdf_directory):
    '''Process all PDF files in a directory'''
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files
    pdf_files = list(pdf_dir.glob("**/*.pdf")) + list(pdf_dir.glob("**/*.PDF"))
    print(f"Found {len(pdf_files)} PDF files to ingest")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # add source information to documents metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages\n")
        except Exception as e:
            print(f" ERROR: {e}\n")
    
    print(f"Total pages loaded {len(all_documents)}")
    # usually PyPDFLoader returns one Document per PDF page
    # the exact number is number of LangChain Document class
    return all_documents

In [3]:
all_pdf_documents = process_all_pdfs("/Users/aaronchen/Documents/Code/learning/data/pdf")

Found 7 PDF files to ingest
Processing: MaterialsHorizons.pdf
 Loaded 11 pages

Processing: chen-et-al-2024-diffusion-limited-crystal-growth-of-gallium-nitride-using-active-machine-learning.pdf
 Loaded 9 pages

Processing: acs.chemmater.9b03142.pdf
 Loaded 8 pages

Processing: zhang2018.pdf
 Loaded 14 pages

Processing: nguyen-et-al-2025-steering-amine-co2-chemistry-a-molecular-insight-into-the-amino-site-relationship-of-carbamate-and.pdf
 Loaded 11 pages

Processing: full published article.pdf
 Loaded 10 pages

Processing: chen-et-al-2023-transferable-force-field-for-gallium-nitride-crystal-growth-from-the-melt-using-on-the-fly-active.pdf
 Loaded 12 pages

Total pages loaded 75


In [4]:
### Text splitting into chuncks

def split_document(documents, chunk_size=1000, chunk_overlap=200):
    '''Split documents into smaller chunks for better RAG performance'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""] # spliting by
        # paragraphs, lines, words, final fallback by individual characters
    )
    split_docs = text_splitter.split_documents(documents)
    for chunk_index, doc in enumerate(split_docs):
        doc.metadata["chunk_index"] = chunk_index
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # show example of a chunk
    if split_docs:
        print(f"Example chunk:")
        print(f"Context: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

chunks = split_document(all_pdf_documents)

Split 75 documents into 530 chunks
Example chunk:
Context: Materials
Horizons
rsc.li/materials-horizons
 COMMUNICATION 
 Paulette Clancy, Barry P. Rand  et al . 
 A comprehensive picture of roughness evolution in organic 
crystalline growth: the role of molec...
Metadata: {'producer': 'iTextSharp™ 5.5.11 ©2000-2017 iText Group NV (AGPL-version)', 'creator': 'PyPDF', 'creationdate': '2022-10-28T14:47:42+05:30', 'author': 'Jordan T. Dull', 'crossmarkdomains[1]': 'rsc.org', 'crossmarkdomainexclusive': 'false', 'crossmarkmajorversiondate': '2022-10-28', 'moddate': '2023-05-01T17:04:14+01:00', 'subject': 'Materials Horizons (2022), 9, 2752-2761, doi:10.1039/D2MH00854H', 'title': 'A comprehensive picture of roughness evolution in organic crystalline growth: the role of molecular aspect ratio', 'doi': '10.1039/D2MH00854H', 'source': '/Users/aaronchen/Documents/Code/learning/data/pdf/MaterialsHorizons.pdf', 'total_pages': 11, 'page': 0, 'page_label': '4', 'source_file': 'MaterialsHorizons.pdf'

### Embedding and Vectore Store DB

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import hashlib
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """ Manage document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load embedding model"""
        try:
            print(f"Loading embeding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error in loading model: {self.model_name} {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate embedings for a list of texts

        Args:
            texts: List of text string to embed
        
        Return:
            numpy array of embedding with shape (len(texts), embedding dims)
        """

        print(f"Generating embedding for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding shape: {embeddings.shape}")
        return embeddings

### initialize embedding manager
embedding_manager = EmbeddingManager()

Loading embeding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9885.89it/s]


Model loaded. Embedding dimension: 384


In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "/Users/aaronchen/Documents/Code/learning/data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            store_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            metadata = dict(doc.metadata)
            # Generate chunk id based on metadata and hash
            source_file = Path(metadata.get("source_file", "unknown")).stem
            page = metadata.get("page", 0)
            chunk_index = metadata.get("chunk_index", i)
            text_hash = hashlib.sha1(doc.page_content.encode("utf-8")).hexdigest()[:10]
            doc_id = f"{source_file}_p{page}_c{chunk_index}_{text_hash}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content) 
            # store page_content for text retrieval later. Otherwise only embdding is stored.
            # Sometimes you can use another text store to keep the original texts
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1590


In [8]:
### Convert text chunks to embeddings
text_chunks = [doc.page_content for doc in chunks]
### Generate embeddings
embeddings = embedding_manager.generate_embedding(text_chunks)
### Store in the Vector DB
vector_store.add_documents(documents=chunks, embeddings=embeddings)

Generating embedding for 530 texts...


Batches: 100%|██████████| 17/17 [00:03<00:00,  4.83it/s]


Generated embedding shape: (530, 384)
Adding 530 documents to vector store...
Successfully added 530 documents to vector store
Total documents in collection: 2120


### Retriever Pipelin from Vector Store

In [9]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs        

        except Exception as e:
            print(f"Error during retrieval: {e}")
            print("Return empty list")
            return []

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

In [10]:
rag_retriever.retrieve('What is FLARE force field')



Retrieving documents for query: 'What is FLARE force field'
Top K: 5, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.41it/s]

Generated embedding shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_43458845_450',
  'content': '23\nThis “mapping” does not manipulate or truncate any existing\ninteractions that arose from the FLARE-generated model. We\nprovide the force field file together with associated parameters\nfor the MD simulations in the Data Availability and Supporting\nInformation.\n22,24\nThis will enable users in the community to\nconduct large-scale MD simulations using the nonparametric\nforce field we developed with FLARE but in the more\ncommonly used format of a LAMMPS model.\n2. COMPUTATIONAL METHODS\nThis section covers: (1) the methodology used to create the\nGP-based force field model through the OTF training and\nfrom ab initio MD trajectories and (2) the process used to\nvalidate the FLARE force field against established properties in\nGaN. A graphical schematic of the workflow for the FLARE\nOTF training is shown in Figure 1. Here, we provide a brief\noverview of the FLARE method, the OTF learning, and the\nGP-mapped force fields. Further detail

### Integrtion Vector DB Context Pipeline with LLM Output

In [11]:
### RAG pipeline with Groq LLM
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
load_dotenv()

### Initialize LLM using OpenRouter free model
llm = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY_RAG"),
    base_url="https://openrouter.ai/api/v1",
    model="openai/gpt-oss-120b:free",
    temperature=0.1,
    max_tokens=1024,
)

# Simple RAG pipeline: retrieve context from DB + generate response
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return "No relevant context is found to answer the question"

    context = "\n\n".join(doc["content"] for doc in results)
    messages = [
        SystemMessage(content="Answer the question concisely using only the retrieved context. If the context is insufficient, say so explicitly."),
        HumanMessage(content=(
            f"Context:\n{context}\n\n"
            f"Question:\n{query}\n\n"
            "Answer:"
        )),
    ]

    response = llm.invoke(messages)
    return response.content

In [13]:
answer = rag_simple("Why do we use machine learning force field for Gallium Nitride crystal growth study?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Why do we use machine learning force field for Gallium Nitride crystal growth study?'
Top K: 3, Score threshold: 0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.04it/s]

Generated embedding shape: (1, 384)
Retrieved 3 documents (after filtering)


We use a machine‑learning force field because existing semi‑empirical models cannot capture the atomic‑scale mechanisms of GaN crystal growth—such as the detailed crystallization at the interface, the preferred faceted growth directions, and nitrogen diffusion/reactivity in the gallium melt. A machine‑learning potential can learn the fundamental interatomic interactions from data and provide a realistic, accurate description of these processes, which experimental methods cannot resolve.


In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:50] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
user_question = "What mechanism governs the organic semiconductor surface roughness " \
"in the scenario of physical vapor deposition "
result = rag_advanced(user_question, rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What mechanism governs the organic semiconductor surface roughness in the scenario of physical vapor deposition '
Top K: 3, Score threshold: 0.1
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.31it/s]

Generated embedding shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: The surface roughness that develops during physical‑vapor‑deposition of the organic semiconductors is governed by **kinetic roughening arising from the balance between ad‑molecule diffusion and nucleation on the growing crystal**.  In this regime the molecular aspect ratio controls the diffusion length and nucleation density, so that molecules with a higher aspect ratio diffuse farther before nucleating, leading to smoother, step‑flow‑like growth, whereas low‑aspect‑ratio molecules nucleate more densely and produce a rougher, island‑growth morphology.  Thus, the roughness evolution is dictated by the diffusion‑limited nucleation (kinetic roughening) mechanism, modulated by the molecular aspect ratio.
Sources: [{'source': 'MaterialsHorizons.pdf', 'page': 1, 'score': 0.25909268856048584, 'preview': '2752 |  Mater. Horiz., 2022, 9, 2752–2761 This jou...'}, {'source': 'MaterialsHorizons.pdf', 'page': 1, 'score': 0.25909268856048584, 'preview': '2752 |  Mater. Horiz., 2022, 9, 2752–